# Machine and Deep Learning · Lecture 5
## Decision trees and ensembles for capstone design
Graduate Diploma in Data Science | Instructor Colab | Two-hour online session

### Learning outcomes
1. Explain how a classification tree chooses a split and how pruning controls complexity.
2. Distinguish bagging, random forests and gradient boosting.
3. Compare a baseline and tree-based candidates using protected evaluation data.
4. Interpret feature importance without making causal claims.
5. Translate a domain idea into a feasible, testable capstone proposal.

**Run code cells in order.** The live demonstration is CPU-friendly and uses a bundled dataset. Student practice and the proposal template are in the separate lab notebook. Pause at each discussion question, invite students to make a prediction before running code, and ask them to explain evidence rather than just name a winning model.

## 01 · From an interesting topic to a prediction task

An **intended user** acts on the output. The **unit of prediction** is one example the model scores. The **prediction moment** is when inputs must be available. The **target** is the result to learn, defined independently of the model. A capstone pitch needs all four, plus a defensible dataset and an evaluation plan.

**Running case:** Given one prepared grayscale image of a handwritten digit, classify it as 0–9 so a hypothetical digitisation operator can review uncertain transcriptions. This is a classroom benchmark for single digits, not a system for reading complete forms.

### Discussion · Turn a broad topic into a task
What would you ask someone who pitches ‘AI to improve student success’? In pairs, identify (1) the intended user and decision, (2) exactly one unit to score, (3) the moment the prediction is made, (4) a target that can be observed without using future information, and (5) what action follows. What harms could arise from a false positive or false negative? What evidence would make this a feasible capstone rather than an aspiration? Share one revised, testable task statement.

## 02 · Inspect the data and protect the test set

Digits contains 1,797 labelled 8 × 8 grayscale images represented by 64 pixel features. **Stratification** keeps approximate class proportions. **Training** fits parameters, **validation** compares settings and models, and **test** estimates performance after decisions are frozen. This exercise uses roughly 60/20/20.

If deployment involves future observations, split by time; if records repeat by person, device, or writer, group by entity when IDs are available. This row split does not establish performance on unseen writers or a new scanner.

In [ ]:
# Purpose: import data, modelling, plotting, splitting, and evaluation utilities used in this lecture.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
try:
    from IPython.display import display
except ImportError:
    display = print
from sklearn.datasets import load_digits, load_wine
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, ConfusionMatrixDisplay, classification_report
from sklearn.inspection import permutation_importance

SEED = 42  # Make randomized splits and estimators reproducible for this teaching run.
print("scikit-learn:", sklearn.__version__)  # Record library version when discussing reproducibility.

In [ ]:
# Purpose: load the bundled digit dataset, build named feature/target tables, inspect data quality, and visualize examples.
d = load_digits()  # No download or upload is required.
X = pd.DataFrame(d.data, columns=[f"r{i//8}c{i%8}" for i in range(64)])  # Each column is one pixel in the 8×8 image.
y = pd.Series(d.target, name="digit")  # The known digit is the prediction target.
print(X.shape, "| missing cells:", X.isna().sum().sum(), "| duplicate rows:", X.duplicated().sum())  # Quick shape and quality check.
fig, axes = plt.subplots(1, 5, figsize=(9, 2))  # Prepare a small visual sample.
for i, ax in enumerate(axes):
    ax.imshow(d.images[i], cmap="gray")  # Display the image corresponding to row i.
    ax.set_title(str(y.iloc[i]))  # Show the known label for visual inspection.
    ax.axis("off")
plt.show()

In [ ]:
# Purpose: reserve a stratified test set, split the remaining development data into train/validation, and audit class balance.
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=.20, stratify=y, random_state=SEED)  # Hold test rows aside before model selection.
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=.25, stratify=y_dev, random_state=SEED)  # 25% of 80% gives roughly 20% overall validation.
assert set(X_train.index).isdisjoint(X_val.index)  # Check train/validation rows do not overlap.
assert set(X_dev.index).isdisjoint(X_test.index)  # Check development and protected test rows do not overlap.
display(pd.DataFrame({"train": y_train.value_counts(),
                      "validation": y_val.value_counts(),
                      "test": y_test.value_counts()}).sort_index())  # Verify class proportions across splits.

### Discussion · Is a random split fair?
Would random rows be a fair split for a project predicting future faults from repeated readings on the same machines? Identify the leakage risk if the same machine appears in train and test. Which split would better match the intended deployment: time-based, machine-grouped, or both? What happens if a machine has no historical records? Explain how the evaluation design should reflect the prediction moment and the population the model will face.

## 03 · How a decision tree learns

The **root** is the first node; an internal **node** asks a question; the **branches** route observations; a **leaf** gives a prediction. The learner greedily chooses a feature and threshold that give the biggest local improvement among the splits considered. **Gini impurity** is one minus the sum of squared class proportions. It is zero for a pure node and 0.50 for an even binary mix.

**Worked split:** A node has 4 A and 4 B. A proposed split yields children (3 A, 1 B) and (1 A, 3 B). Both child impurities are 0.375. Their size-weighted average is 0.375, so the reduction is 0.125. The split is useful without being perfect.

In [ ]:
# Purpose: fit a deliberately shallow tree and visualize its questions, branches, and predicted leaves.
illustration = DecisionTreeClassifier(max_depth=2, random_state=SEED).fit(X_train, y_train)  # Restrict depth so the tree is readable.
plt.figure(figsize=(15, 6))
plot_tree(illustration, feature_names=X.columns.tolist(), filled=True, fontsize=8)  # Color and label nodes for classroom inspection.
plt.show()

### Discussion · Evaluate a split
Would a split that creates one pure leaf and one very large mixed leaf always be the best choice? Use the size-weighted impurity idea to explain why purity alone is insufficient. What might happen if the pure leaf contains only one unusual training observation? How would you judge the split using validation data and the costs of different errors? Inspect the plotted tree: which questions are easy to explain, and what information would a domain expert need before trusting them?

## 04 · Overfitting and pruning

**Depth** measures the longest root-to-leaf path in edges. **Leaf count** measures the number of terminal rules. A large train–validation gap is a warning, not a proof on its own. **Pre-pruning** limits growth, e.g. maximum depth or minimum leaf size. **Post-pruning** removes branches from a grown tree. Cost-complexity pruning trades training impurity against leaf count; increasing **ccp_alpha** makes extra leaves more costly. Too much pruning can underfit. Choose settings on validation; alpha zero is allowed and means no cost-complexity pruning.

In [ ]:
# Purpose: define a consistent macro-F1 scorer and compare tree complexity settings on training and validation data.
def macro_f1(actual, pred):
    return f1_score(actual, pred, average="macro", zero_division=0)  # Give every digit class equal weight.

def tree_trials(configs):
    rows = []  # Store one summary row per tree configuration.
    for label, kwargs in configs.items():
        model = DecisionTreeClassifier(random_state=SEED, **kwargs).fit(X_train, y_train)  # Fit only on training rows.
        rows.append({"setting": label, "depth": model.get_depth(),
                     "leaves": model.get_n_leaves(),
                     "train_f1": macro_f1(y_train, model.predict(X_train)),
                     "val_f1": macro_f1(y_val, model.predict(X_val)), **kwargs})  # Record complexity and both scores.
    return pd.DataFrame(rows)

depths = tree_trials({"Unrestricted": {}, "Depth 2": {"max_depth": 2},
                      "Depth 5": {"max_depth": 5}})  # Contrast a flexible tree with constrained alternatives.
display(depths[["setting","depth","leaves","train_f1","val_f1"]].round(3))  # Compare fit and generalization evidence.

In [ ]:
# Purpose: evaluate a small predeclared cost-complexity pruning grid on validation macro-F1 and freeze one alpha.
ALPHAS = [0, .001, .003, .01, .03]  # Candidate pruning strengths; zero means no pruning penalty.
pruning = tree_trials({f"alpha={a}": {"ccp_alpha": a} for a in ALPHAS})  # Fit each setting on train and score on validation.
pruning = pruning.sort_values(["val_f1", "leaves", "ccp_alpha"],
                              ascending=[False, True, False])  # Prefer score, then simpler tree for exact ties.
chosen_alpha = float(pruning.iloc[0]["ccp_alpha"])  # Freeze the selected setting before test evaluation.
display(pruning.round(3))
print("Frozen alpha for the pruning candidate:", chosen_alpha)

### Discussion · Complexity versus generalization
The unrestricted tree scores 1.00 on training and 0.78 on validation. A smaller tree scores 0.91 and 0.83. Which would you advance and why? What does the gap suggest, and what can’t it prove? Consider whether a small validation difference is meaningful, whether the simpler model is easier to explain, and whether the error costs are symmetric. Would you keep testing settings on the test set? Why not?

## 05 · Bagging and random forests

**Bagging** fits members independently on bootstrap samples (rows drawn with replacement), then aggregates their predictions. Duplicated draws mean some original rows are absent from a member’s sample. Averaging diverse, unstable trees can reduce variance. A **random forest** also considers a random subset of features at each split, encouraging diversity among trees. For these scikit-learn classifiers, prediction aggregates class probabilities rather than necessarily taking a hard vote.

### Discussion · Why does an ensemble help?
If all 80 trees always make the same mistakes, what does averaging give us? Explain why diversity matters alongside the number of trees. What are the benefits and costs of training many trees compared with a single shallow tree? In what setting might speed, memory, or interpretability matter more than a small score gain?

### Break
After the break, compare independently trained trees with sequential corrections. Before returning, try to recall which ensemble method uses bootstrap samples and which adds models in sequence.

## 06 · Gradient boosting

Boosting is **sequential**. Each stage adds small trees aimed at improving the current ensemble under a **loss function**. In regression with squared-error loss, the next tree fits residuals (actual minus predicted). In classification, gradient boosting fits a correction based on the negative gradient of the classification loss, not literal subtraction of class labels. **Learning rate** shrinks each stage’s contribution; **number of stages** and **tree depth** affect complexity.

### Discussion · Compare ensemble strategies
Which method uses independently trained members, and which is a sequence of corrections? In your own words, explain what bagging tries to stabilize and what boosting tries to improve stage by stage. What could go wrong if boosting is allowed too many stages or overly deep trees? How might learning rate interact with the number of stages? Identify a practical reason to choose one approach beyond validation score.

## 07 · Compare candidates and freeze the choice

A **baseline** is a reference that the candidate must improve on in a meaningful way. The dummy predicts the most frequent training class. Accuracy counts all correct cases; **macro F1** averages per-class F1 equally, appropriate for this teaching task where each digit matters. A per-class F1 balances precision and recall.

Use the same training and validation rows and a declared metric for all models. Freeze the highest validation macro F1 candidate, breaking exact ties by the original order in the candidate dictionary for reproducibility. Do not treat tiny differences as reliable superiority. The validation table is for selection and is not an unbiased estimate of the selected model’s future performance.

In [ ]:
# Purpose: train a baseline and four tree/ensemble candidates, compare validation performance, and freeze a candidate.
candidates = {
    "Dummy baseline": DummyClassifier(strategy="most_frequent"),  # Establish a simple reference score.
    "Unrestricted tree": DecisionTreeClassifier(random_state=SEED),
    "Pruning candidate": DecisionTreeClassifier(ccp_alpha=chosen_alpha, random_state=SEED),
    "Bagging": BaggingClassifier(n_estimators=40, random_state=SEED),
    "Random forest": RandomForestClassifier(n_estimators=80, max_features="sqrt", random_state=SEED),
    "Gradient boosting": GradientBoostingClassifier(
        n_estimators=35, learning_rate=.1, max_depth=2, random_state=SEED),
}
rows = []  # Collect comparable metrics for each candidate.
for priority, (name, model) in enumerate(candidates.items()):
    model.fit(X_train, y_train)  # Every candidate sees the same training partition.
    pv = model.predict(X_val)  # Validation predictions support model selection, not final unbiased reporting.
    rows.append({"model": name, "val_macro_f1": macro_f1(y_val, pv),
                 "val_accuracy": accuracy_score(y_val, pv),
                 "train_macro_f1": macro_f1(y_train, model.predict(X_train)),
                 "tie_priority": priority})  # Track train/validation gap and deterministic tie order.
results = pd.DataFrame(rows).sort_values(
    ["val_macro_f1", "tie_priority"], ascending=[False, True])  # Rank by the declared selection metric.
winner = results.iloc[0]["model"]  # Freeze the highest-scoring candidate; test data remain untouched.
display(results.drop(columns="tie_priority").round(3))
print("Frozen candidate:", winner)

### Discussion · Small score differences
Your forest scores 0.901 and your shallow tree scores 0.899. What would you write in a proposal or report? Would you claim the forest is better? Discuss sampling variation, validation-set size, simplicity, speed, interpretability, and the intended user. What additional evaluation would you plan if the choice mattered in practice? State why the test set should not be repeatedly consulted to settle this comparison.

## 08 · Feature importance and final evaluation

**Impurity importance** aggregates training split improvements attributed to each feature; it can favour variables with many possible splits. **Permutation importance** measures how the selected score changes when one column is shuffled for a fitted model. This demo interprets a predeclared forest on validation data, not necessarily the winning model. Correlated features can substitute for each other; negative or near-zero scores need caution. Neither ranking establishes causation or permission to use the feature.

The frozen winner is refit on training plus validation and assessed once on test. Interpret class-level errors from the confusion matrix and report limits such as domain shift or unavailable writer IDs.

In [ ]:
# Purpose: compare forest impurity importance with validation-set permutation importance.
forest = candidates["Random forest"]  # This is a predeclared forest, which may differ from the selected winner.
pi = permutation_importance(forest, X_val, y_val, scoring="f1_macro",
                            n_repeats=5, random_state=SEED)  # Shuffle each feature repeatedly and measure score change.
imp = pd.DataFrame({"impurity_importance": forest.feature_importances_,
                    "mean_validation_f1_drop": pi.importances_mean,
                    "shuffle_sd": pi.importances_std}, index=X.columns)  # Align each importance estimate to its pixel.
display(imp.sort_values("impurity_importance", ascending=False).head(8).round(4))  # Show top split-based features.
display(imp.sort_values("mean_validation_f1_drop", ascending=False).head(8).round(4))  # Show features whose shuffle most reduces validation F1.

In [ ]:
# Purpose: refit the frozen candidate on all development data, then evaluate once on the protected test set.
from sklearn.base import clone  # Make a fresh unfitted copy of the selected estimator.
final_model = clone(candidates[winner]).fit(X_dev, y_dev)  # Use train plus validation after selection is complete.
test_pred = final_model.predict(X_test)  # Obtain test predictions once for final evaluation.
print(winner, "| test macro F1:", round(macro_f1(y_test, test_pred), 3),
      "| test accuracy:", round(accuracy_score(y_test, test_pred), 3))
print(classification_report(y_test, test_pred, zero_division=0))  # Inspect precision, recall and F1 by digit.
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, cmap="Purples")  # Show which classes are confused.
plt.title("Held-out test set: frozen candidate")
plt.tight_layout()
plt.show()

### Discussion · Interpret importance and error
If a customer ID ranks highly, what would you investigate before keeping it? Consider whether it is available at prediction time, a proxy for identity or time, a source of leakage, or simply a shortcut tied to the sampled population. How would correlated features affect permutation importance? Looking at the confusion matrix, which errors matter most to the intended user, and what information would be needed to judge that? Why does feature importance not show that changing a feature would change the prediction or outcome?

## 09 · Turn the workflow into a capstone proposal

**Suggested submission:** a one-page idea brief, not a completed system. Use the seven headings in the separate lab notebook.

### Worked capstone idea · Handwritten digit transcription
1. **Decision:** Help a digitisation operator identify which single-digit entries to review. The model is for an offline feasibility study, not automated acceptance.
2. **Task:** At transcription time, classify one prepared 8 × 8 grayscale digit image as 0–9. The label is its recorded digit, not a feature.
3. **Data:** Start with the bundled Digits benchmark and its pixel inputs. It supports a classroom comparison. If scanner-specific data are unavailable, keep the claim at benchmark level.
4. **Models:** Use a majority dummy and a shallow tree as references; compare a random forest as a candidate for nonlinear pixel interactions. A later neural model would need its own justified comparison.
5. **Evaluation:** Use shared stratified development partitions and a protected test split; choose by macro F1. Inspect class-level confusion. For a claim about new writers or scanners, collect IDs and reserve appropriate groups.
6. **Risk:** Small benchmark images are not whole forms. Domain shift could produce confident errors. Keep human review and do not claim deployment readiness.
7. **Milestone:** Produce a reproducible data audit, split report and baseline on development data by the next session.

### Transfer to your domain
For a **fault warning**, define the warning horizon and consider future-period or machine-group holdout; missed faults and false alarms have different costs. For **price prediction**, a median predictor is a useful reference and mean absolute error reports average error in price units. For **vision or text**, compare a defensible simple baseline before adding a neural model, and split repeated or related examples together. If labels are unavailable, revise the question or justify a different task and its evaluation.

### Discussion · One-sentence capstone pitch
In one sentence, what does your model predict, using what information, and what action could follow? Then test the sentence against these prompts: who uses the prediction; what is one unit; when are features available; how is the target defined; what baseline will you compare against; how will you split data to match deployment; and what could go wrong? Exchange pitches with a partner and identify one ambiguity or unsupported claim to revise.

### References
- [Module specification and weekly plan](https://chatgpt.com/library?file_id=libfile_affae89c59f881919bd874e07d2eee1a&account_id=personal)
- [Decision trees and cost-complexity pruning](https://scikit-learn.org/stable/modules/tree.html)
- [Bagging, random forests and gradient boosting](https://scikit-learn.org/stable/modules/ensemble.html)
- [Permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html)
- [Digits dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)
- [Wine dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html)